# Cancer Type Classification Using RNA-Seq Gene Expression Data

## Overview

This project applies machine learning techniques to classify five cancer types using high-dimensional RNA-Seq gene expression data. Due to the complexity of gene expression profiles, dimensionality reduction techniques such as PCA, t-SNE, and UMAP are used to analyze structure and visualize class separability.

Supervised learning models are then trained to classify cancer types, and feature importance analysis is performed to identify key genes contributing to predictions.

## Objectives

- Classify cancer types using gene expression data.
- Reduce high-dimensional gene space using PCA.
- Visualize data structure using t-SNE and UMAP.
- Compare multiple machine learning models.
- Identify important genes contributing to classification.

## Dataset Description

- 801 patient samples
- ~20,000 gene expression features
- 5 cancer types:
  - BRCA
  - KIRC
  - COAD
  - LUAD
  - PRAD

## Methodology

The workflow consists of the following stages:

1. Data preprocessing and cleaning
2. Feature standardization
3. Dimensionality reduction using PCA
4. Non-linear visualization using t-SNE and UMAP
5. Supervised classification using multiple machine learning models
6. Model evaluation using accuracy and cross-validation
7. Model interpretability using feature importance analysis

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt
import umap

In [ ]:
# load data
X = pd.read_csv("dataset/data.csv")
y = pd.read_csv("dataset/labels.csv")

## Data Cleaning

The first column contains sample identifiers and is removed as it does not contribute to prediction. Missing values are also checked.

In [ ]:
# data cleaning
y = y.iloc[:, -1].values.ravel()

print(X.shape, y.shape)

print(X.isnull().sum().sum())

X = X.drop(X.columns[0], axis=1)

## Train-Test Split

The dataset is split into training and testing sets using an 80:20 ratio with stratification to preserve class distribution.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Feature Scaling

Standardization is applied to normalize gene expression values for PCA and machine learning models.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Dimensionality Reduction Strategy

Due to the extremely high dimensionality of gene expression data, multiple dimensionality reduction techniques are applied:

- PCA is used for linear compression while preserving global variance.
- t-SNE is used to capture local neighborhood structure for visualization.
- UMAP is used to preserve both local and global structure, often producing more biologically meaningful clustering.

These methods are not used for classification, but strictly for exploratory data analysis and visualization.

In [ ]:
pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(sum(pca.explained_variance_ratio_))

## PCA Visualization

PCA does not clearly separate cancer classes, indicating that the data structure is non-linear.

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=y_train)
plt.title("PCA Visualization")
plt.show()

## t-SNE Visualization

t-SNE reveals strong clustering of cancer types, indicating non-linear separability in gene expression space.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_train_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_train)
plt.title("t-SNE Visualization")
plt.show()

## UMAP Visualization

UMAP provides clearer and more structured clustering compared to PCA and t-SNE, preserving both local and global relationships.

In [ ]:
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_train_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=y_train)
plt.title("UMAP Visualization")
plt.show()

## Visualization Insights

PCA projection shows weak class separability, indicating that cancer gene expression patterns are not linearly separable.

In contrast, both t-SNE and UMAP reveal well-separated clusters of cancer types, suggesting that the underlying data manifold is highly non-linear and structured.

UMAP provides more globally consistent clustering compared to t-SNE.

## Model Training

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_pca, y_train)
pred = lr.predict(X_test_pca)
print("LR Accuracy:", accuracy_score(y_test, pred))

In [24]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_pca, y_train)
pred = rf.predict(X_test_pca)
print("RF Accuracy:", accuracy_score(y_test, pred))

RF Accuracy: 0.968944099378882


In [ ]:
# SVM
svm = SVC(kernel='rbf')
svm.fit(X_train_pca, y_train)
pred = svm.predict(X_test_pca)
print("SVM Accuracy:", accuracy_score(y_test, pred))

In [ ]:
# Confusion Matrix for SVM

cm = confusion_matrix(y_test, pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - SVM")
plt.show()

## Gene Importance Analysis

Random Forest is trained on original scaled gene expression data to identify important genes contributing to cancer classification.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train_scaled, y_train)

importances = rf.feature_importances_

genes = X.columns

feature_importance = pd.DataFrame({
    "Gene": genes,
    "Importance": importances
})

top_genes = feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

print(top_genes)

In [ ]:
# Cross Validation
scores = cross_val_score(lr, X_train_pca, y_train, cv=5)

print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

## Model Explainability using SHAP

To interpret the trained machine learning model, SHAP (SHapley Additive exPlanations) is used to quantify feature contributions to predictions.

In [ ]:
import shap

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train_scaled, y_train)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_train_scaled)

shap.summary_plot(shap_values, X_train, feature_names=X.columns)

## Conclusion

This project successfully classified five cancer types using RNA-Seq gene expression data.

While PCA provided a linear projection of the data, it did not clearly separate cancer classes, indicating that the underlying structure is non-linear.

t-SNE and UMAP visualizations revealed strong and meaningful clustering of cancer types, confirming that gene expression data contains distinct biological patterns.

Machine learning models, particularly SVM and Logistic Regression, achieved high classification accuracy. Additionally, Random Forest feature importance identified key genes that may act as potential biomarkers for cancer classification.

Overall, the combination of dimensionality reduction, classification models, and feature importance analysis demonstrates that gene expression data is highly effective for distinguishing cancer types.